# SMD Analysis

Goal:
- Test if the current implementation of the SMD is correct or not
- Fix a reliable SMD method

The SMD (Street Mover's Distance) or Wasserstein distance is a distance metric between point clouds. It has the following properties:

1) Given a point cloud A in $\mathbb{R}^n$ and a translation vector $t\in\mathbb{R}^n$ the SMD between A and its translate A+t is
$$
d_{SMD}(A, A+t) = |t|_2
$$
2) SMD is linear in scale: Given two point clouds A and B and a scale factor $\lambda>0$ then:
$$
d_{SMD}(\lambda A, \lambda B) = \lambda d_{SMD}(A,B)
$$

We test different implementations wrt to correctness and runtimes as follows:
- We generate a random point cloud A in 3D with N points.
- We generate $B = A + t$, a translated point cloud having translation distance $T = |t|_2$ from A.
- We compute the distance between A and B.
- We compute the distance between $S*A$ and $S*B$ where S is a sclaing factor.



## POT package implementation

Satisfies these two properties:

In [9]:
import numpy as np
import ot

N = 100 # Number nodes
T = 7 # Translation distance
S = 0.2 # Scaling factor

# Generate two 3D point clouds A and A + t
coords = np.random.rand(N, 3)
translated_coords = [coord + np.array([T,0,0]) for coord in coords]

scaled_coords_G = [coord * S for coord in coords]
scaled_coords_H = [coord * S for coord in translated_coords]

def to_array(coords):
    return np.array([coord for coord in coords])

def ws_dist(A, B):
    A = to_array(A)
    B = to_array(B)
    # Compute pairwise distance matrix
    M = ot.dist(A, B, metric='euclidean')

    # Uniform weights (assume each point contributes equally)
    w_A = np.ones(len(A)) / len(A)
    w_B = np.ones(len(B)) / len(B)

    # Compute Wasserstein distance (Optimal Transport)
    wasserstein_distance = ot.emd2(w_A, w_B, M)
    return wasserstein_distance


print(f"WS distance: {ws_dist(coords, translated_coords)} (Translation distance: {T})")
print(f"WS distance: {ws_dist(scaled_coords_G, scaled_coords_H)} (Scaled translation distance: {S*T})")


WS distance: 7.000000000000009 (Translation distance: 7)
WS distance: 1.400000000000001 (Scaled translation distance: 1.4000000000000001)


## SMD implementation taken from Vesselformer

We compute the Sinkhorn distance between two translated and scaled point clouds 1) with the Sinkhorn implementation from Vesselformer and 2) witht the implementation from POT package.

NOTE: Sinkhorn distance is an efficient approximation to the Wasserstein distance.

RESULT: The POT implementation returns exakt distance, while Vesselformer returns incorrect values.

In [10]:
import numpy as np
from project.streetmover_distance import SinkhornDistance
import torch

N = 100 # Number nodes
T = 20 # Translation distance
S = 0.4 # Scaling factor

# Generate two 3D point clouds A and A + t
coords = np.random.rand(N, 3)
translated_coords = [coord + np.array([T,0,0]) for coord in coords]

scaled_coords_G = [coord * S for coord in coords]
scaled_coords_H = [coord * S for coord in translated_coords]

def to_array(coords):
    return np.array([coord for coord in coords])

def smd_dist(A, B):
    A = to_array(A)
    B = to_array(B)
    A = torch.FloatTensor(A)
    B = torch.FloatTensor(B)

    sinkhorn_distance = SinkhornDistance(eps=1e-7, max_iter=100, reduction='none')
    cost, pi, C = sinkhorn_distance(A, B)

    return cost

def smd_dist_from_ot(A, B):
    A = to_array(A)
    B = to_array(B)
    # Compute pairwise distance matrix
    M = ot.dist(A, B, metric='euclidean')

    # Uniform weights (assume each point contributes equally)
    w_A = np.ones(len(A)) / len(A)
    w_B = np.ones(len(B)) / len(B)
    # Compute Sinkhorn Distance (Regularized Optimal Transport)
    lambda_reg = 0.05  # Regularization parameter
    S_dist = ot.sinkhorn2(w_A, w_B, M, lambda_reg)
    return S_dist


print(f"Sinkhorn distance: {smd_dist(coords, translated_coords)} (Translation distance: {T})")
print(f"Sinkhorn distance: {smd_dist(scaled_coords_G, scaled_coords_H)} (Scaled translation distance: {S*T})")
print("\n")

print(f"Sinkhorn distance (from POT): {smd_dist_from_ot(coords, translated_coords)} (Translation distance: {T})")
print(f"Sinkhorn distance (from POT): {smd_dist_from_ot(scaled_coords_G, scaled_coords_H)} (Scaled translation distance: {S*T})")


Sinkhorn distance: 30651.380859375 (Translation distance: 20)
Sinkhorn distance: 53.77177429199219 (Scaled translation distance: 8.0)


Sinkhorn distance (from POT): 20.007059924906237 (Translation distance: 20)
Sinkhorn distance (from POT): 8.002963947402163 (Scaled translation distance: 8.0)


## Runtimes of Wasserstein vs Sinkhorn distance

In [11]:
import time 

N = 10000 # Number nodes

T = 20 # Translation distance
S = 0.4 # Scaling factor

# Generate two 3D point clouds A and A + t
coords = np.random.rand(N, 3)
translated_coords = [coord + np.array([T,0,0]) for coord in coords]

scaled_coords_G = [coord * S for coord in coords]
scaled_coords_H = [coord * S for coord in translated_coords]

start = time.time()
print(f"WS distance: {ws_dist(coords, translated_coords)} (Translation distance: {T})")
print(f"WS distance: {ws_dist(scaled_coords_G, scaled_coords_H)} (Scaled translation distance: {S*T})")
run1 = time.time() - start
start = time.time()
print(f"Sinkhorn distance (from POT): {smd_dist_from_ot(coords, translated_coords)} (Translation distance: {T})")
print(f"Sinkhorn distance (from POT): {smd_dist_from_ot(scaled_coords_G, scaled_coords_H)} (Scaled translation distance: {S*T})")
run2 = time.time() - start

print(f"\nRuntimes for {N} points:")
print(f"Wasserstein dist: {run1} s")
print(f"Sinkhorn dist: {run2} s")

c:\Users\ckarg\.local\share\micromamba\envs\graphs\Lib\site-packages\ot\lp\__init__.py:630: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  check_result(result_code)


WS distance: 20.000247644372184 (Translation distance: 20)
WS distance: 8.000099057748889 (Scaled translation distance: 8.0)
Sinkhorn distance (from POT): 20.00767827032621 (Translation distance: 20)
Sinkhorn distance (from POT): 8.003238417439745 (Scaled translation distance: 8.0)

Runtimes for 10000 points:
Wasserstein dist: 18.008758783340454 s
Sinkhorn dist: 7.247912645339966 s
